# 캡스톤 모듈 M1 — 주파수 보정 곁가지 (QFC-lite) 학습과 평가

**목표:** 카메라·코덱은 그대로 두고, 고정된 YOLO26s의 **neck P3(stride 8)** 에 작은 곁가지를 더해 압축으로 잃은 작은 객체 탐지를 되살린다.

**설계 근거 (사전실험 C·D)**
- 입력 1280: 640은 압축 손실을 가리고(C), 특징 변화와 미탐의 연결도 약함(D)
- 보충할 정보: 1280에서 회복 여지가 큰 Mid+High 대역(C)
- 넣는 위치: 미탐과 가장 직결된 P3(D). 8×8 DCT(stride 8)와 P3의 해상도가 같아 그대로 더할 수 있음

**구조:** 압축 프레임 → 휘도 8×8 DCT(고정) → 대역 선택 → 경량 CNN(depthwise-separable) → 0으로 초기화한 1×1 conv → P3에 더함. YOLO26s 가중치는 전부 고정.

**학습 (라벨 불필요):** 같은 프레임의 **무손실 원본을 넣은 같은 탐지기를 교사**로 쓴다.
- `L_feat`: 보정된 P3를 원본 프레임의 P3에 맞춤 (D의 drift를 직접 줄임)
- `L_out`: P3 레벨 탐지 출력(클래스·박스)을 원본 프레임의 출력에 맞춤 (탐지 손실 역할)
- 원본 프레임도 학습에 섞어서, 깨끗한 입력에서는 보정값이 0에 가깝게 유지되도록 함

**비교 (변형 3개를 같은 조건으로 학습)**
| 이름 | 곁가지 입력 | 답하는 질문 |
|---|---|---|
| `dct_midhigh` | DCT Mid+High (u+v≥3, 58채널) | 제안 방법 |
| `dct_all` | DCT 전 대역 (64채널) | 대역 선택이 필요한가 |
| `pixel` | 8×8 픽셀을 그대로 펼친 것 (64채널, DCT 없음) | 주파수 분해가 필요한가 |

기준: 무보정(Baseline), 원본(Reference), **오라클**(압축 프레임의 Mid+High 계수를 원본으로 교체, C와 같은 방식. 모듈이 도달할 수 있는 상한의 기준)

**데이터 분할 (시퀀스 단위):** 평가 = HighwayView, CampusView (A·C·D의 CCTV 주평가 묶음). 학습 = 나머지 시퀀스. 평가 장면은 학습에 한 번도 쓰지 않음.

**산출 (`M1_out/`)**: `M1_ap.csv`(APs·APm·APl), `M1_summary.csv`(격차 회복률·오라클 대비), `M1_object_recovery.csv`(놓친 작은 객체 회복·손해, bootstrap), `M1_efficiency.csv`(파라미터·지연시간), `M1_train_log.csv`, `qfc_*.pt`, 그림

**실행:** Input에 `uvg-build-dataset`(uvg_cctv), GPU T4, Internet On → `TEST_RUN=True`(약 10분)로 Run All → `False`로 Save & Run All (**약 2~2.5시간**)

**알려진 한계:** 압축 재학습 비교군은 이 노트북에 없음(탐지기 전체 학습이 필요해 별도). 휘도만 사용. 학습 장면 10개 안팎. 교사가 원본 프레임의 탐지기라서 "원본에서도 못 찾는 객체"는 되살릴 수 없음.


In [1]:
!pip install -q ultralytics==8.4.165 pycocotools

In [2]:
%%writefile cd_lib.py
"""사전실험 E1(순서) · E2(디블로킹) 공통 라이브러리"""
import json, os, subprocess, contextlib, io
import numpy as np, cv2

W, H = 1280, 720
CCTV_CLASSES = [1, 2, 3, 4, 6, 8]   # person, bicycle, car, motorcycle, bus, truck (COCO 순서 + 1)
BALL_CLASSES = [33]

# ---------------- 디코딩 / 인코딩 ----------------
def read_yuv(video, w, h, keep=None, vf=None):
    """yuv420p 그대로 디코딩. keep=None이면 전체 프레임 리스트, 아니면 {i: (y,u,v)}"""
    cmd = ["ffmpeg", "-v", "error", "-i", video]
    if vf: cmd += ["-vf", vf]
    cmd += ["-f", "rawvideo", "-pix_fmt", "yuv420p", "-"]
    fb = w * h * 3 // 2
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE)
    out = [] if keep is None else {}
    i = 0
    while True:
        b = p.stdout.read(fb)
        if len(b) < fb: break
        if keep is None or i in keep:
            a = np.frombuffer(b, np.uint8)
            f = (a[:w*h].reshape(h, w).copy(), a[w*h:w*h*5//4].reshape(h//2, w//2).copy(),
                 a[w*h*5//4:].reshape(h//2, w//2).copy())
            if keep is None: out.append(f)
            else: out[i] = f
        i += 1
    p.wait()
    if p.returncode: raise RuntimeError(f"decode failed: {video}")
    return out, i

def resize_yuv(f, w2, h2):
    """같은 함수(INTER_AREA)로 두 경로 모두 축소 -> 순서 외의 차이 제거"""
    y, u, v = f
    return (cv2.resize(y, (w2, h2), interpolation=cv2.INTER_AREA),
            cv2.resize(u, (w2//2, h2//2), interpolation=cv2.INTER_AREA),
            cv2.resize(v, (w2//2, h2//2), interpolation=cv2.INTER_AREA))

def yuv_to_bgr(f):
    y, u, v = f; h, w = y.shape
    buf = np.concatenate([y.ravel(), u.ravel(), v.ravel()]).reshape(h * 3 // 2, w)
    return cv2.cvtColor(buf, cv2.COLOR_YUV2BGR_I420)

def encode(frames, w, h, out, mode, val, x264_params=None, fps=30):
    """CCTV형 x264 (veryfast, zerolatency, B프레임 없음, GOP 60). mode: 'kbps'(1-pass CBR형) 또는 'qp'(고정 QP)"""
    cmd = ["ffmpeg", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w}x{h}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "veryfast", "-tune", "zerolatency",
           "-profile:v", "high", "-pix_fmt", "yuv420p", "-g", "60", "-keyint_min", "60", "-sc_threshold", "0"]
    if mode == "kbps":
        cmd += ["-b:v", f"{val}k", "-maxrate", f"{val}k", "-bufsize", f"{val}k"]
    else:
        cmd += ["-qp", str(val)]
    if x264_params: cmd += ["-x264-params", x264_params]
    cmd += ["-an", out]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    for (y, u, v) in frames:
        p.stdin.write(y.tobytes()); p.stdin.write(u.tobytes()); p.stdin.write(v.tobytes())
    p.stdin.close(); p.wait()
    if p.returncode: raise RuntimeError(f"encode failed: {out}")

def kbps_of(p):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=bit_rate", "-of", "csv=p=0", p],
                       capture_output=True, text=True).stdout.strip()
    return round(int(r) / 1000, 1) if r.isdigit() else None

def has_filter(name):
    r = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True).stdout
    return any(l.split()[1:2] == [name] for l in r.splitlines() if len(l.split()) > 1)

def psnr(a, b):
    mse = float(((a.astype(np.float64) - b.astype(np.float64)) ** 2).mean())
    return 99.0 if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)

# ---------------- GT ----------------
def build_gt(raw, n, names):
    images, anns, aid = [], [], 0
    for i in range(n):
        images.append({"id": i, "width": W, "height": H})
        for o in raw.get(str(2 * i + 1), []):
            x1, y1, x2, y2 = o["x_min"] * W, o["y_min"] * H, o["x_max"] * W, o["y_max"] * H
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0: continue
            aid += 1
            anns.append({"id": aid, "image_id": i, "category_id": int(o["class_id"]), "bbox": [x1, y1, w, h],
                         "area": w * h, "iscrowd": int(o.get("iscrowd", 0)), "track_id": o.get("track_id")})
    return {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": n_} for i, n_ in enumerate(names)]}

def size_class(a):
    return "small" if a < 32 ** 2 else ("medium" if a < 96 ** 2 else "large")

# ---------------- 매칭 (B-3·C와 같은 규칙) ----------------
def match_detected(gt_boxes, gt_cls, pred, conf_thr=0.25, iou_thr=0.5):
    det = np.zeros(len(gt_boxes), bool)
    if len(gt_boxes) == 0 or len(pred) == 0: return det
    p = pred[pred[:, 4] >= conf_thr]
    if len(p) == 0: return det
    p = p[np.argsort(-p[:, 4])]
    g = gt_boxes
    gx2, gy2, px2, py2 = g[:, 0] + g[:, 2], g[:, 1] + g[:, 3], p[:, 0] + p[:, 2], p[:, 1] + p[:, 3]
    iw = np.clip(np.minimum(gx2[:, None], px2[None]) - np.maximum(g[:, 0][:, None], p[:, 0][None]), 0, None)
    ih = np.clip(np.minimum(gy2[:, None], py2[None]) - np.maximum(g[:, 1][:, None], p[:, 1][None]), 0, None)
    inter = iw * ih
    iou = inter / np.maximum((g[:, 2] * g[:, 3])[:, None] + (p[:, 2] * p[:, 3])[None] - inter, 1e-9)
    iou = np.where(gt_cls[:, None] == p[None, :, 5], iou, 0.0)
    for j in range(len(p)):
        col = np.where(det, -1.0, iou[:, j]); k = int(np.argmax(col))
        if col[k] >= iou_thr: det[k] = True
    return det

def coco_ap(gt, preds, classes):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    if not preds: return None
    with contextlib.redirect_stdout(io.StringIO()):
        cg = COCO(); cg.dataset = gt; cg.createIndex(); cd = cg.loadRes(preds)
        E = COCOeval(cg, cd, "bbox"); E.params.catIds = classes
        E.evaluate(); E.accumulate(); E.summarize()
    f = lambda x: None if x < 0 else round(100 * x, 2)
    s = E.stats
    return {"AP": f(s[0]), "AP50": f(s[1]), "APs": f(s[3]), "APm": f(s[4]), "APl": f(s[5]), "ARs": f(s[9])}

def cluster_boot(cid, fn, B=1000, seed=0):
    rng = np.random.default_rng(seed); K = int(cid.max()) + 1
    point = fn(np.ones(len(cid))); vals = []
    for _ in range(B):
        cnt = np.bincount(rng.integers(0, K, K), minlength=K)
        v = fn(cnt[cid].astype(float))
        if v is not None and np.isfinite(v): vals.append(v)
    if not vals: return point, np.nan, np.nan
    return point, float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# ---------------- E3: PyAV로 디코딩하면서 매크로블록 QP 지도와 프레임 유형 추출 ----------------
def decode_with_qp(video, keep, w=1280, h=720):
    """반환: {i: (y,u,v)}, {i: qp_map (h/16, w/16)}, {i: 'I'|'P'|'B'}"""
    import av
    c = av.open(video); s = c.streams.video[0]
    s.codec_context.options = {"export_side_data": "venc_params"}
    T = av.sidedata.sidedata.Type
    frames, qps, ptypes = {}, {}, {}
    for i, fr in enumerate(c.decode(s)):
        if i not in keep: continue
        sd = fr.side_data.get(T.VIDEO_ENC_PARAMS)
        if sd is None: raise RuntimeError("QP side data 없음: PyAV/FFmpeg 버전 확인")
        qps[i] = np.asarray(sd.qp_map(), dtype=np.float32)
        ptypes[i] = {1: "I", 2: "P", 3: "B"}.get(int(fr.pict_type), str(fr.pict_type))
        a = fr.to_ndarray(format="yuv420p")            # (h*3/2, w)
        frames[i] = (a[:h].copy(), a[h:h + h // 4].reshape(h // 2, w // 2).copy(),
                     a[h + h // 4:].reshape(h // 2, w // 2).copy())
    c.close()
    return frames, qps, ptypes

def box_mb(b, mbw=80, mbh=45, mb=16):
    x, y, w, h = b
    x0, y0 = int(max(0, x // mb)), int(max(0, y // mb))
    x1, y1 = int(min(mbw, -(-(x + w) // mb))), int(min(mbh, -(-(y + h) // mb)))
    return x0, max(x0 + 1, x1), y0, max(y0 + 1, y1)

def qp_features(q, b):
    """객체 매크로블록 평균·최대 QP, 같은 행 배경(객체 열 제외) 평균 QP, 프레임 평균 QP"""
    x0, x1, y0, y1 = box_mb(b, q.shape[1], q.shape[0])
    obj = q[y0:y1, x0:x1]
    rows = q[y0:y1].copy(); rows[:, x0:x1] = np.nan
    bg = np.nanmean(rows) if np.isfinite(rows).any() else np.nan
    return float(obj.mean()), float(obj.max()), float(bg), float(q.mean())

# ---------------- E4: 레이트 컨트롤 설정별 인코딩, 전체 프레임 디코딩 ----------------
def has_encoder(name):
    r = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
    return any(len(l.split()) > 1 and l.split()[1] == name for l in r.splitlines())

def encode_args(frames, w, h, out, vargs, fps=30):
    """vargs: 코덱·레이트 컨트롤 인자 리스트 (예: ['-c:v','libx264',...])"""
    cmd = ["ffmpeg", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-"] + vargs + ["-an", out]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    for (y, u, v) in frames:
        p.stdin.write(y.tobytes()); p.stdin.write(u.tobytes()); p.stdin.write(v.tobytes())
    p.stdin.close(); p.wait()
    if p.returncode: raise RuntimeError(f"encode failed: {out}")

def decode_all_with_qp(video, w=1280, h=720):
    """전체 프레임: (yuv 리스트, 프레임 평균 QP 배열(없으면 NaN), 프레임 유형 리스트, 패킷 바이트 배열)"""
    import av
    c = av.open(video); s = c.streams.video[0]
    s.codec_context.options = {"export_side_data": "venc_params"}
    T = av.sidedata.sidedata.Type
    yuv, qp, pt, nb = [], [], [], []
    for pkt in c.demux(s):
        size = pkt.size
        for fr in pkt.decode():
            sd = fr.side_data.get(T.VIDEO_ENC_PARAMS)
            qp.append(float(np.asarray(sd.qp_map()).mean()) if sd is not None else np.nan)
            pt.append({1: "I", 2: "P", 3: "B"}.get(int(fr.pict_type), str(fr.pict_type)))
            nb.append(size)
            a = fr.to_ndarray(format="yuv420p")
            yuv.append((a[:h].copy(), a[h:h + h // 4].reshape(h // 2, w // 2).copy(), a[h + h // 4:].reshape(h // 2, w // 2).copy()))
    c.close()
    return yuv, np.array(qp), pt, np.array(nb)

# ---------------- E6: 색차 처리 ----------------
GRAY_U = np.full((360, 640), 128, np.uint8)

def gray_bgr_from_y(y):
    return yuv_to_bgr((y, GRAY_U, GRAY_U))

def bgr_to_uv(bgr):
    """BGR -> I420의 U, V (yuv_to_bgr과 같은 BT.601 limited range의 역변환)"""
    import cv2
    yuv = cv2.cvtColor(bgr, cv2.COLOR_BGR2YUV_I420)
    h, w = bgr.shape[:2]
    u = yuv[h:h + h // 4].reshape(h // 2, w // 2); v = yuv[h + h // 4:].reshape(h // 2, w // 2)
    return u.copy(), v.copy()

def chroma_pearson(u1, v1, u2, v2):
    a = np.concatenate([u1.ravel(), v1.ravel()]).astype(np.float64)
    b = np.concatenate([u2.ravel(), v2.ravel()]).astype(np.float64)
    if a.std() == 0 or b.std() == 0: return 0.0
    return float(np.corrcoef(a, b)[0, 1])

# ---------------- E7: 색차 업샘플링 (모든 조건에 같은 BT.601 limited 변환) ----------------
def yuv444_to_bgr(y, U, V):
    """y: (H,W) uint8, U,V: (H,W) float/uint8 전해상도 색차 -> BGR uint8. OpenCV I420과 같은 계수"""
    Y = 1.164 * (y.astype(np.float32) - 16.0); u = U.astype(np.float32) - 128.0; v = V.astype(np.float32) - 128.0
    r = Y + 1.596 * v; g = Y - 0.813 * v - 0.391 * u; b = Y + 2.018 * u
    return np.clip(np.dstack([b, g, r]) + 0.5, 0, 255).astype(np.uint8)

def up_nearest(c):
    return np.repeat(np.repeat(c, 2, axis=0), 2, axis=1).astype(np.float32)

def up_bilinear(c):
    import cv2
    h, w = c.shape
    return cv2.resize(c.astype(np.float32), (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)

def _box(x, r):
    import cv2
    return cv2.boxFilter(x, -1, (2 * r + 1, 2 * r + 1), borderType=cv2.BORDER_REFLECT)

def guided_filter(I, p, r, eps):
    """He et al. 가이드 필터. I: 길잡이(휘도), p: 입력(업샘플된 색차), 값 범위 0~1"""
    mI, mp = _box(I, r), _box(p, r)
    vI = _box(I * I, r) - mI * mI; cIp = _box(I * p, r) - mI * mp
    a = cIp / (vI + eps); b = mp - a * mI
    return _box(a, r) * I + _box(b, r)

def up_guided(y, c, r=4, eps=1e-3):
    I = y.astype(np.float32) / 255.0; p = up_bilinear(c) / 255.0
    return np.clip(guided_filter(I, p, r, eps), 0, 1) * 255.0

def up_lm(y, c, r=2, eps=1e-3):
    """저해상도에서 색차 = a·휘도 + b 국소 선형 모델을 맞추고, 원해상도 휘도에 적용 (빠른 가이드 업샘플링, CCLM과 같은 원리)"""
    import cv2
    h, w = c.shape
    Il = cv2.resize(y, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0; pl = c.astype(np.float32) / 255.0
    mI, mp = _box(Il, r), _box(pl, r)
    vI = _box(Il * Il, r) - mI * mI; cIp = _box(Il * pl, r) - mI * mp
    a = cIp / (vI + eps); b = mp - a * mI
    a, b = _box(a, r), _box(b, r)
    A = cv2.resize(a, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR); B = cv2.resize(b, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)
    return np.clip(A * (y.astype(np.float32) / 255.0) + B, 0, 1) * 255.0

# ---------------- 실험 C: 8x8 블록 DCT 대역 개입 ----------------
from scipy.fft import dctn, idctn
def bdct(y):
    return dctn(y.reshape(H // 8, 8, W // 8, 8).transpose(0, 2, 1, 3), axes=(2, 3), norm="ortho")
def ibdct(c):
    return idctn(c, axes=(2, 3), norm="ortho").transpose(0, 2, 1, 3).reshape(H, W)
_UV = np.add.outer(np.arange(8), np.arange(8))
BAND = {"Low": (_UV >= 1) & (_UV <= 2), "Mid": (_UV >= 3) & (_UV <= 6), "High": _UV >= 7}
def band_replace(Dc, Dr, band):
    out = Dc.copy(); m = BAND[band]; out[..., m] = Dr[..., m]; return ibdct(out)
def band_remove(Dc, band):
    out = Dc.copy(); out[..., BAND[band]] = 0.0; return ibdct(out)
def letterbox_params(imgsz, stride=32):
    r = min(imgsz / H, imgsz / W); nw, nh = round(W * r), round(H * r)
    return r, (-(-nw // stride) * stride - nw) / 2, (-(-nh // stride) * stride - nh) / 2, -(-nh // stride) * stride

def to_u8(y):
    return np.clip(np.rint(y), 0, 255).astype(np.uint8)


Writing cd_lib.py


In [3]:
# ===== 설정 =====
TEST_RUN = False
VARIANTS = ["dct_midhigh", "dct_all", "pixel"]
TEST_SEQS = ["HighwayView", "CampusView"]            # 평가 전용 (학습에 쓰지 않음)
ALL_SEQS = ["HighwayView", "CampusView", "JobFair", "OfficeWalk", "TrafficLights", "SpotRobot", "BoatReverse",
            "VolleyballGame", "FloorballTrain", "FloorballGame", "WinterDrive", "HighwayDrive"]
KBPS = [1000, 500, 250]
EPOCHS, BATCH, TRAIN_STEP, EVAL_STEP, N_BOOT = 8, 8, 3, 5, 1000
LR, WD = 2e-3, 1e-4
W_FEAT, W_CLS, W_BOX, W_POS = 1.0, 1.0, 1.0, 20.0     # 손실 가중치 (W_POS: 교사가 객체라고 본 위치의 가중)
WIDTH, DEPTH = 128, 4                                  # 곁가지 채널 수, 블록 수
CONF_MATCH, IOU_MATCH = 0.25, 0.5; SEED = 0; N_LIMIT = None
if TEST_RUN: EPOCHS, TRAIN_STEP, N_LIMIT, N_BOOT, VARIANTS = 1, 10, 40, 30, ["dct_midhigh", "pixel"]
GOP = 60
X264 = ["-c:v", "libx264", "-preset", "veryfast", "-tune", "zerolatency", "-profile:v", "high", "-pix_fmt", "yuv420p",
        "-g", str(GOP), "-keyint_min", str(GOP), "-sc_threshold", "0"]
def cbr(k): return X264 + ["-b:v", f"{k}k", "-maxrate", f"{k}k", "-bufsize", f"{k}k"]   # A-3과 같은 CCTV형 설정

import os, sys, json, glob, time, shutil, random
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, cv2
sys.path.insert(0, os.getcwd()); import cd_lib as L
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if not TEST_RUN: assert torch.cuda.is_available(), "GPU 없음 -> Accelerator GPU T4"
OUT = os.path.abspath("M1_out"); ENC = os.path.abspath("enc_m1"); os.makedirs(OUT, exist_ok=True); os.makedirs(ENC, exist_ok=True)
cand = glob.glob("/kaggle/input/**/uvg_cctv", recursive=True) + ([os.environ["UVG_ROOT"]] if os.environ.get("UVG_ROOT") else [])
assert cand, "Input에 uvg_cctv(데이터셋 구축 노트북 Output) 없음 -> Add Input에서 uvg-build-dataset 추가"
REF_ROOT = cand[0]
SEQS = [s for s in ALL_SEQS if os.path.exists(f"{REF_ROOT}/{s}/ref_720p30.mkv")]
TRAIN_SEQS = [s for s in SEQS if s not in TEST_SEQS]
assert all(s in SEQS for s in TEST_SEQS) and TRAIN_SEQS, f"시퀀스 확인: {SEQS}"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("uvg_cctv:", REF_ROOT, "\n학습:", TRAIN_SEQS, "\n평가:", TEST_SEQS, "| 장치", DEV)


uvg_cctv: /kaggle/input/notebooks/naeunku/uvg-build-dataset/uvg_cctv 
학습: ['JobFair', 'OfficeWalk', 'TrafficLights', 'SpotRobot', 'BoatReverse', 'VolleyballGame', 'FloorballTrain', 'FloorballGame', 'WinterDrive', 'HighwayDrive'] 
평가: ['HighwayView', 'CampusView'] | 장치 cuda


In [4]:
# ===== 데이터: 압축(1000·500·250 kbps) -> 필요한 프레임만 메모리에 (YUV 4:2:0 그대로) =====
from ultralytics import YOLO
M = YOLO("yolo26s.pt"); NAMES = [M.names[i] for i in range(80)]
DATA = {}; t0 = time.time()
for si, s in enumerate(SEQS):
    n = json.load(open(f"{REF_ROOT}/{s}/done.json"))["ref_frames"]
    if N_LIMIT: n = min(n, N_LIMIT)
    is_test = s in TEST_SEQS
    keep = list(range(0, n, EVAL_STEP if is_test else TRAIN_STEP))
    ref_all, _ = L.read_yuv(f"{REF_ROOT}/{s}/ref_720p30.mkv", 1280, 720)
    ref_all = ref_all[:n]; frames = {"ref": {i: ref_all[i] for i in keep}}; rate = {}
    for k in KBPS:
        p = f"{ENC}/{s}_{k}.mp4"; L.encode_args(ref_all, 1280, 720, p, cbr(k))
        frames[k], _ = L.read_yuv(p, 1280, 720, keep=set(keep)); rate[k] = L.kbps_of(p); os.remove(p)
    del ref_all
    d = {"si": si, "test": is_test, "keep": keep, "frames": frames}
    if is_test:
        gt = L.build_gt(json.load(open(f"{REF_ROOT}/{s}/ann_raw.json")), n, NAMES)
        anns = {i: [a for a in gt["annotations"] if a["image_id"] == i and a["category_id"] in L.CCTV_CLASSES] for i in keep}
        d.update({"gt": gt, "anns": anns, "gb": {i: np.array([a["bbox"] for a in anns[i]], float).reshape(-1, 4) for i in keep},
                  "gc": {i: np.array([a["category_id"] for a in anns[i]]) for i in keep}})
    DATA[s] = d
    print(f"{s:15s} {'평가' if is_test else '학습'} 프레임 {len(keep):3d} | 실측 kbps {rate} ({time.time()-t0:.0f}s)", flush=True)
shutil.rmtree(ENC, ignore_errors=True)
TRAIN_ITEMS = [(s, i, v) for s in TRAIN_SEQS for i in DATA[s]["keep"] for v in ["ref"] + KBPS]
print("학습 표본(프레임 × 버전):", len(TRAIN_ITEMS), "| 평가 프레임:", sum(len(DATA[s]["keep"]) for s in TEST_SEQS))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
HighwayView     평가 프레임  60 | 실측 kbps {1000: 940.1, 500: 463.3, 250: 225.1} (12s)
CampusView      평가 프레임  60 | 실측 kbps {1000: 986.5, 500: 483.8, 250: 234.8} (24s)
JobFair         학습 프레임 100 | 실측 kbps {1000: 959.7, 500: 465.1, 250: 226.4} (36s)
OfficeWalk      학습 프레임 100 | 실측 kbps {1000: 928.8, 500: 488.3, 250: 259.4} (49s)
TrafficLights   학습 프레임 100 | 실측 kbps {1000: 977.6, 500: 480.7, 250: 237.8} (61s)
SpotRobot       학습 프레임 100 | 실측 kbps {1000: 1017.6, 500: 507.1, 250: 255.3} (73s)
BoatReverse     학습 프레임  50 | 실측 kbps {1000: 1069.7, 500: 514.4, 250: 241.0} (80s)
VolleyballGame  학습 프레임 100 | 실측 kbps {1000: 968.6, 500: 484.7, 250: 243.3} (91s)
FloorballTrain  학습 프레임 100 | 실측 kbps {1000: 1017.5, 500: 

In [5]:
# ===== 탐지기(고정)와 곁가지 =====
PAD = 8                                                    # 720 -> 736 (위아래 8px, Ultralytics letterbox와 같음. 8의 배수라 8x8 격자 유지)
def letterbox(bgr):
    return cv2.copyMakeBorder(bgr, PAD, PAD, 0, 0, cv2.BORDER_CONSTANT, value=(114, 114, 114))
def to_tensor(bgrs):                                       # Ultralytics 전처리와 같게: BGR->RGB, 0~1
    x = torch.from_numpy(np.stack(bgrs)).to(DEV); return x[..., [2, 1, 0]].permute(0, 3, 1, 2).float().div(255.0)

_ = M.predict(np.zeros((720, 1280, 3), np.uint8), imgsz=1280, device=0 if DEV == "cuda" else "cpu", verbose=False)
NET = M.predictor.model.model if hasattr(M.predictor.model, "model") else M.model      # 실제 추론에 쓰이는 모델
NET.eval()
for p in NET.parameters(): p.requires_grad_(False)
HEAD = NET.model[-1]

def dct_basis():
    k = np.arange(8)[:, None]; x = np.arange(8)[None, :]
    T = np.cos(np.pi * (2 * x + 1) * k / 16); T[0] *= np.sqrt(1 / 8); T[1:] *= np.sqrt(2 / 8)
    B = np.einsum("ui,vj->uvij", T, T).reshape(64, 1, 8, 8)            # 채널 u*8+v
    return torch.tensor(B, dtype=torch.float32), (np.add.outer(np.arange(8), np.arange(8))).reshape(-1)
class DSBlock(nn.Module):
    def __init__(s, c):
        super().__init__(); s.dw = nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False); s.pw = nn.Conv2d(c, c, 1, bias=False); s.bn = nn.BatchNorm2d(c); s.act = nn.SiLU()
    def forward(s, x): return x + s.act(s.bn(s.pw(s.dw(x))))
class QFC(nn.Module):
    """입력: 탐지기 입력 텐서(RGB 0~1) -> 출력: P3에 더할 보정값"""
    def __init__(s, kind, c_out, width=128, depth=4):
        super().__init__(); s.kind = kind
        B, uv = dct_basis(); s.register_buffer("basis", B)
        idx = np.arange(64) if kind != "dct_midhigh" else np.where(uv >= 3)[0]
        s.register_buffer("idx", torch.tensor(idx)); cin = len(idx)
        s.norm = nn.BatchNorm2d(cin); s.stem = nn.Sequential(nn.Conv2d(cin, width, 1, bias=False), nn.BatchNorm2d(width), nn.SiLU())
        s.body = nn.Sequential(*[DSBlock(width) for _ in range(depth)])
        s.out = nn.Conv2d(width, c_out, 1); nn.init.zeros_(s.out.weight); nn.init.zeros_(s.out.bias)      # 0 초기화: 처음엔 원래 탐지기와 동일
    def forward(s, x):
        y = (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]).float()
        f = F.pixel_unshuffle(y, 8) if s.kind == "pixel" else F.conv2d(y, s.basis, stride=8)
        return s.out(s.body(s.stem(s.norm(f[:, s.idx]))))

# hook: 첫 층 입력에서 보정값을 계산하고, Detect 입력의 P3에 더한다 (predict에서도 그대로 동작)
STATE = {"branch": None, "delta": None, "p3": None, "grab": False}
def _first(mod, inp):
    if STATE["branch"] is not None:
        with torch.no_grad(): STATE["delta"] = STATE["branch"](inp[0])
def _detect(mod, inp):
    feats = list(inp[0])
    if STATE["grab"]: STATE["p3"] = feats[0]
    if STATE["branch"] is not None and STATE["delta"] is not None:
        feats[0] = feats[0] + STATE["delta"].to(feats[0].dtype); return (feats,)
NET.model[0].register_forward_pre_hook(_first); HEAD.register_forward_pre_hook(_detect)

# 확인 1: P3 채널 수와 해상도 / 확인 2: 0 초기화 곁가지를 달아도 출력이 완전히 같음 / 확인 3: DCT 기저가 scipy와 일치
STATE["grab"] = True; x0 = to_tensor([letterbox(np.random.default_rng(0).integers(0, 255, (720, 1280, 3), np.uint8).astype(np.uint8))])
with torch.no_grad(): o_ref = NET(x0)[0].clone()
C_P3 = STATE["p3"].shape[1]; assert STATE["p3"].shape[2:] == (92, 160), STATE["p3"].shape
STATE["branch"] = QFC("dct_midhigh", C_P3, WIDTH, DEPTH).to(DEV).eval()
with torch.no_grad(): o_new = NET(x0)[0]
assert torch.equal(o_ref, o_new), "0 초기화 곁가지가 출력을 바꿈"
yy = np.random.default_rng(1).random((720, 1280)); B_, _ = dct_basis()
c_t = F.conv2d(torch.tensor(yy, dtype=torch.float32)[None, None], B_, stride=8)[0].numpy().reshape(8, 8, 90, 160).transpose(2, 3, 0, 1)
assert np.allclose(c_t, L.bdct(yy), atol=1e-4), "DCT 기저 불일치"
n_det = sum(p.numel() for p in NET.parameters()); n_br = sum(p.numel() for p in STATE["branch"].parameters())
STATE["branch"] = None; STATE["grab"] = False
print(f"P3 채널 {C_P3} | 탐지기 파라미터 {n_det/1e6:.2f}M | 곁가지 {n_br/1e3:.1f}K (+{100*n_br/n_det:.2f}%) | 확인 3개 통과")


P3 채널 128 | 탐지기 파라미터 9.50M | 곁가지 95.5K (+1.01%) | 확인 3개 통과


In [6]:
# ===== 학습: 원본 프레임을 넣은 같은 탐지기가 교사 (라벨 불필요) =====
class Pairs(torch.utils.data.Dataset):
    def __len__(s): return len(TRAIN_ITEMS)
    def __getitem__(s, j):
        q, i, v = TRAIN_ITEMS[j]; fr = DATA[q]["frames"]
        a = letterbox(L.yuv_to_bgr(fr["ref"][i])); b = a if v == "ref" else letterbox(L.yuv_to_bgr(fr[v][i]))
        if random.random() < 0.5: a = a[:, ::-1]; b = b[:, ::-1]          # 좌우 반전 (1280은 8의 배수라 블록 격자 유지)
        return torch.from_numpy(np.ascontiguousarray(a)), torch.from_numpy(np.ascontiguousarray(b))
def prep(u8): return u8.to(DEV)[..., [2, 1, 0]].permute(0, 3, 1, 2).float().div(255.0)
def features_p3(x):
    """고정 탐지기를 통과시켜 Detect 입력 P3를 꺼냄 (곁가지 없이)"""
    br = STATE["branch"]; STATE["branch"] = None; STATE["grab"] = True
    with torch.no_grad(), torch.autocast("cuda", enabled=(DEV == "cuda")): NET(x)      # 고정 부분은 반정밀도로 빠르게
    STATE["grab"] = False; STATE["branch"] = br; return STATE["p3"].float()
def head_p3(p3):
    return HEAD.cv2[0](p3), HEAD.cv3[0](p3)                                # 박스(4채널), 클래스 logit(80채널)
def train(kind):
    torch.manual_seed(SEED); random.seed(SEED)
    br = QFC(kind, C_P3, WIDTH, DEPTH).to(DEV); opt = torch.optim.AdamW(br.parameters(), lr=LR, weight_decay=WD)
    dl = torch.utils.data.DataLoader(Pairs(), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
    steps = EPOCHS * len(dl); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=max(steps, 2), pct_start=0.1)
    log = []; t0 = time.time(); it = 0
    for ep in range(EPOCHS):
        br.train(); agg = np.zeros(4); nb = 0
        for a, b in dl:
            xa, xb = prep(a), prep(b); n = xa.shape[0]
            p3 = features_p3(torch.cat([xa, xb])); p3_ref, p3_cmp = p3[:n], p3[n:]
            with torch.no_grad(): t_box, t_cls = head_p3(p3_ref); t_prob = t_cls.sigmoid(); w = 1 + W_POS * t_prob.amax(1, keepdim=True)
            p3s = p3_cmp + br(xb); s_box, s_cls = head_p3(p3s)
            l_feat = F.mse_loss(p3s, p3_ref) / (p3_ref.pow(2).mean() + 1e-8)
            l_cls = (F.binary_cross_entropy_with_logits(s_cls, t_prob, reduction="none").sum(1, keepdim=True) * w).mean()
            l_box = ((s_box - t_box).abs().mean(1, keepdim=True) * t_prob.amax(1, keepdim=True)).sum() / (t_prob.amax(1, keepdim=True).sum() + 1e-6)
            loss = W_FEAT * l_feat + W_CLS * l_cls + W_BOX * l_box
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(br.parameters(), 5.0); opt.step()
            if it < steps - 1: sched.step()
            it += 1; agg += [loss.item(), l_feat.item(), l_cls.item(), l_box.item()]; nb += 1
        agg /= max(nb, 1); log.append({"variant": kind, "epoch": ep + 1, "loss": agg[0], "l_feat": agg[1], "l_cls": agg[2], "l_box": agg[3], "min": (time.time() - t0) / 60})
        print(f"{kind:12s} ep {ep+1}/{EPOCHS} loss {agg[0]:.4f} (feat {agg[1]:.4f} cls {agg[2]:.4f} box {agg[3]:.4f}) {(time.time()-t0)/60:.1f}분", flush=True)
    br.eval(); torch.save(br.state_dict(), f"{OUT}/qfc_{kind}.pt"); return br, log
BR, LOG = {}, []
for kind in VARIANTS:
    BR[kind], lg = train(kind); LOG += lg
pd.DataFrame(LOG).to_csv(f"{OUT}/M1_train_log.csv", index=False)


dct_midhigh  ep 1/8 loss 0.4318 (feat 0.1260 cls 0.0215 box 0.2844) 2.7분
dct_midhigh  ep 2/8 loss 0.4243 (feat 0.1258 cls 0.0205 box 0.2780) 5.6분
dct_midhigh  ep 3/8 loss 0.4237 (feat 0.1259 cls 0.0203 box 0.2775) 8.4분
dct_midhigh  ep 4/8 loss 0.4211 (feat 0.1260 cls 0.0200 box 0.2751) 11.3분
dct_midhigh  ep 5/8 loss 0.4170 (feat 0.1261 cls 0.0197 box 0.2712) 14.2분
dct_midhigh  ep 6/8 loss 0.4122 (feat 0.1261 cls 0.0195 box 0.2666) 17.1분
dct_midhigh  ep 7/8 loss 0.4097 (feat 0.1262 cls 0.0195 box 0.2640) 19.9분
dct_midhigh  ep 8/8 loss 0.4062 (feat 0.1260 cls 0.0194 box 0.2608) 22.8분
dct_all      ep 1/8 loss 0.4328 (feat 0.1265 cls 0.0213 box 0.2850) 2.9분
dct_all      ep 2/8 loss 0.4241 (feat 0.1271 cls 0.0199 box 0.2771) 5.8분
dct_all      ep 3/8 loss 0.4158 (feat 0.1271 cls 0.0194 box 0.2693) 8.7분
dct_all      ep 4/8 loss 0.4085 (feat 0.1273 cls 0.0190 box 0.2622) 11.6분
dct_all      ep 5/8 loss 0.4001 (feat 0.1275 cls 0.0187 box 0.2539) 14.5분
dct_all      ep 6/8 loss 0.3955 (feat 0.1273

In [7]:
# ===== 평가: 평가 시퀀스, 입력 1280. Baseline / Oracle(Mid+High 교체) / 변형들 / Reference =====
def predict(imgs):
    out = []
    for s in range(0, len(imgs), 8):
        for r in M.predict(imgs[s:s + 8], imgsz=1280, conf=0.001, max_det=300, device=0 if DEV == "cuda" else "cpu", verbose=False, batch=len(imgs[s:s + 8])):
            b = r.boxes; xy = b.xyxy.cpu().numpy()
            out.append(np.column_stack([xy[:, 0], xy[:, 1], xy[:, 2] - xy[:, 0], xy[:, 3] - xy[:, 1], b.conf.cpu().numpy(), b.cls.cpu().numpy() + 1]))
    return out
def oracle_frame(fc, fr):
    Dc, Dr = L.bdct(fc[0].astype(np.float64)), L.bdct(fr[0].astype(np.float64)); m = L.BAND["Mid"] | L.BAND["High"]
    Dc[..., m] = Dr[..., m]; return (L.to_u8(L.ibdct(Dc)), fc[1], fc[2])
PRED, FLAG = {}, {}; t0 = time.time()
CONDS = ["Baseline", "Oracle"] + VARIANTS
for s in TEST_SEQS:
    d = DATA[s]; keep = d["keep"]
    for lv0 in ["ref"] + KBPS:
        lv = str(lv0)
        for cond in (["Baseline"] + VARIANTS if lv == "ref" else CONDS):
            if cond == "Oracle": imgs = [L.yuv_to_bgr(oracle_frame(d["frames"][lv0][i], d["frames"]["ref"][i])) for i in keep]
            else: imgs = [L.yuv_to_bgr(d["frames"][lv0][i]) for i in keep]
            STATE["branch"] = BR.get(cond); preds = predict(imgs); STATE["branch"] = None
            PRED.setdefault((lv, cond), []).extend((d["si"] * 100000 + i, p) for i, p in zip(keep, preds))
            FLAG.setdefault((lv, cond), {})[s] = np.concatenate([L.match_detected(d["gb"][i], d["gc"][i], p, CONF_MATCH, IOU_MATCH) for i, p in zip(keep, preds)] or [np.zeros(0, bool)])
    print(f"{s} 평가 완료 ({time.time()-t0:.0f}s)", flush=True)
# 합친 GT
g = {"images": [], "annotations": [], "categories": DATA[TEST_SEQS[0]]["gt"]["categories"]}; aid = 0
for s in TEST_SEQS:
    d = DATA[s]; ks = set(d["keep"])
    g["images"] += [{**im, "id": d["si"] * 100000 + im["id"]} for im in d["gt"]["images"] if im["id"] in ks]
    for a in d["gt"]["annotations"]:
        if a["image_id"] in ks and a["category_id"] in L.CCTV_CLASSES:
            aid += 1; g["annotations"].append({**a, "id": aid, "image_id": d["si"] * 100000 + a["image_id"]})
rows = []
for (lv, cond), lst in PRED.items():
    preds = [{"image_id": int(u), "category_id": int(r[5]), "bbox": [float(x) for x in r[:4]], "score": float(r[4])} for u, arr in lst for r in arr if int(r[5]) in L.CCTV_CLASSES]
    m = L.coco_ap(g, preds, L.CCTV_CLASSES)
    if m: rows.append({"input": lv, "cond": cond, **m})
ap = pd.DataFrame(rows); ap.to_csv(f"{OUT}/M1_ap.csv", index=False)
pd.set_option("display.width", 250)
print(ap.pivot_table(index="input", columns="cond", values="APs").reindex(["ref"] + [str(k) for k in KBPS])[[c for c in CONDS if c in ap.cond.unique()]].round(2))


HighwayView 평가 완료 (54s)
CampusView 평가 완료 (109s)
cond   Baseline  Oracle  dct_midhigh  dct_all  pixel
input                                               
ref       43.11     NaN        42.98    42.85  41.43
1000      32.63   37.81        32.57    31.98  31.46
500       26.27   36.81        26.14    25.47  25.49
250       11.93   32.32        11.95    11.50  12.23


In [8]:
# ===== 요약: 격차 회복률, 오라클 대비, 객체 단위 회복·손해, 효율 =====
A = ap.set_index(["input", "cond"]); ref_aps = A.loc[("ref", "Baseline"), "APs"]; summ = []
for k0 in KBPS:
    k = str(k0); base = A.loc[(k, "Baseline"), "APs"]; orc = A.loc[(k, "Oracle"), "APs"]
    for c in VARIANTS + ["Oracle"]:
        v = A.loc[(k, c), "APs"]
        summ.append({"kbps": k, "cond": c, "APs_base": base, "APs": v, "APs_ref": ref_aps, "gain_pp": round(v - base, 2),
                     "gap_recovered_%": round(100 * (v - base) / (ref_aps - base), 1) if ref_aps != base else np.nan,
                     "of_oracle_%": round(100 * (v - base) / (orc - base), 1) if orc != base else np.nan,
                     "APm_gain": A.loc[(k, c), "APm"] - A.loc[(k, "Baseline"), "APm"]})
for c in VARIANTS:   # 원본 입력에서 손해가 없는지
    summ.append({"kbps": "ref", "cond": c, "APs_base": ref_aps, "APs": A.loc[("ref", c), "APs"], "APs_ref": ref_aps, "gain_pp": round(A.loc[("ref", c), "APs"] - ref_aps, 2)})
summ = pd.DataFrame(summ); summ.to_csv(f"{OUT}/M1_summary.csv", index=False); print(summ.to_string(index=False))
# 객체 단위 (작은 객체): 원본에서 찾았고 압축에서 놓친 객체 중 다시 찾은 비율 / 압축에서도 찾던 객체 중 잃은 비율
OBJ = pd.DataFrame([{"seq": s, "frame": i, "area": a["area"], "size": L.size_class(a["area"]),
                     "track": f"{s}:{a['track_id'] if a['track_id'] is not None else 'a' + str(a['id'])}"}
                    for s in TEST_SEQS for i in DATA[s]["keep"] for a in DATA[s]["anns"][i]])
col = lambda lv, c: np.concatenate([FLAG[(lv, c)][s] for s in TEST_SEQS]).astype(float)
rec = []; sm = (OBJ["size"] == "small").to_numpy(); tr = OBJ.track.astype("category").cat.codes.to_numpy()
refd = col("ref", "Baseline")
for k0 in KBPS:
    k = str(k0); base = col(k, "Baseline"); miss = (refd == 1) & (base == 0) & sm; kept = (refd == 1) & (base == 1) & sm
    for c in VARIANTS + ["Oracle"]:
        x = col(k, c)
        r_ = L.cluster_boot(tr, lambda w, x=x: (w * miss * x).sum() / max((w * miss).sum(), 1e-9), B=N_BOOT)
        h_ = L.cluster_boot(tr, lambda w, x=x: (w * kept * (1 - x)).sum() / max((w * kept).sum(), 1e-9), B=N_BOOT)
        n_ = L.cluster_boot(tr, lambda w, x=x: (w * sm * (refd == 1) * (x - base)).sum() / max((w * sm * (refd == 1)).sum(), 1e-9), B=N_BOOT)
        rec.append({"kbps": k, "cond": c, "n_missed": int(miss.sum()), "recovered_%": 100 * r_[0], "rec_lo": 100 * r_[1], "rec_hi": 100 * r_[2],
                    "n_kept": int(kept.sum()), "lost_%": 100 * h_[0], "lost_lo": 100 * h_[1], "lost_hi": 100 * h_[2],
                    "net_survival_pp": 100 * n_[0], "net_lo": 100 * n_[1], "net_hi": 100 * n_[2]})
rec = pd.DataFrame(rec); rec.to_csv(f"{OUT}/M1_object_recovery.csv", index=False); print(rec.round(1).to_string(index=False))
# 효율: 파라미터, 지연시간 (배치 1, 1280 입력)
def latency(branch, n=60):
    STATE["branch"] = branch; x = to_tensor([letterbox(np.zeros((720, 1280, 3), np.uint8))]); ts = []
    with torch.no_grad():
        for j in range(n + 10):
            if DEV == "cuda": torch.cuda.synchronize()
            t = time.perf_counter(); NET(x)
            if DEV == "cuda": torch.cuda.synchronize()
            ts.append(time.perf_counter() - t)
    STATE["branch"] = None; return 1000 * float(np.median(ts[10:]))
n_det = sum(p.numel() for p in NET.parameters()); l0 = latency(None, 8 if TEST_RUN else 60); eff = [{"cond": "Baseline", "params_added": 0, "params_added_%": 0.0, "latency_ms": l0, "latency_added_%": 0.0}]
for c in VARIANTS:
    nb_ = sum(p.numel() for p in BR[c].parameters()); l1 = latency(BR[c], 8 if TEST_RUN else 60)
    eff.append({"cond": c, "params_added": nb_, "params_added_%": 100 * nb_ / n_det, "latency_ms": l1, "latency_added_%": 100 * (l1 - l0) / l0})
eff = pd.DataFrame(eff); eff.to_csv(f"{OUT}/M1_efficiency.csv", index=False); print(eff.round(2).to_string(index=False))
print("\n목표 확인 (제안서): 파라미터 +10% 이하, 지연시간 +20% 이하, 저비트레이트 APs +2%p")


kbps        cond  APs_base   APs  APs_ref  gain_pp  gap_recovered_%  of_oracle_%  APm_gain
1000 dct_midhigh     32.63 32.57    43.11    -0.06             -0.6         -1.2     -0.06
1000     dct_all     32.63 31.98    43.11    -0.65             -6.2        -12.5      0.07
1000       pixel     32.63 31.46    43.11    -1.17            -11.2        -22.6      0.18
1000      Oracle     32.63 37.81    43.11     5.18             49.4        100.0      2.23
 500 dct_midhigh     26.27 26.14    43.11    -0.13             -0.8         -1.2      0.41
 500     dct_all     26.27 25.47    43.11    -0.80             -4.8         -7.6      0.28
 500       pixel     26.27 25.49    43.11    -0.78             -4.6         -7.4      0.67
 500      Oracle     26.27 36.81    43.11    10.54             62.6        100.0      6.81
 250 dct_midhigh     11.93 11.95    43.11     0.02              0.1          0.1      0.87
 250     dct_all     11.93 11.50    43.11    -0.43             -1.4         -2.1      0.41

In [9]:
# ===== 그림 (영어 라벨) =====
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
order = ["Baseline"] + VARIANTS + ["Oracle"]; cols = ["#898781", "#2a78d6", "#eb6834", "#1baf7a", "#52514e"]
fig, ax = plt.subplots(figsize=(7, 3.4)); wd = 0.8 / len(order)
for j, c in enumerate(order):
    v = [A.loc[(str(k), c), "APs"] for k in KBPS]; ax.bar(np.arange(len(KBPS)) + (j - (len(order) - 1) / 2) * wd, v, wd * 0.92, label=c, color=cols[j % len(cols)])
ax.axhline(ref_aps, color="k", lw=0.8, ls="--"); ax.text(len(KBPS) - 0.5, ref_aps, " uncompressed", va="bottom", ha="right", fontsize=8)
ax.set_xticks(range(len(KBPS))); ax.set_xticklabels([f"{k} kbps" for k in KBPS]); ax.set_ylabel("APs (small objects), 1280 input"); ax.legend(fontsize=7, ncol=len(order)); ax.grid(alpha=.3, axis="y")
fig.tight_layout(); fig.savefig(f"{OUT}/M1_fig_APs.png", dpi=130); plt.close(fig)
lg = pd.DataFrame(LOG); fig, ax = plt.subplots(figsize=(5, 3))
for c, gdf in lg.groupby("variant"): ax.plot(gdf.epoch, gdf.loss, "o-", label=c)
ax.set_xlabel("epoch"); ax.set_ylabel("training loss"); ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(f"{OUT}/M1_fig_loss.png", dpi=130); plt.close(fig)
print(sorted(os.listdir(OUT)))


['M1_ap.csv', 'M1_efficiency.csv', 'M1_fig_APs.png', 'M1_fig_loss.png', 'M1_object_recovery.csv', 'M1_summary.csv', 'M1_train_log.csv', 'qfc_dct_all.pt', 'qfc_dct_midhigh.pt', 'qfc_pixel.pt']
